In [1]:
import pandas as pd
from nlp4bia.datasets.Dataset import Dataset
import os

dist_path = "/gpfs/projects/bsc14/storage/corpus/distemist_sp_ca_en_fr_it_ro_po"

In [2]:
url = "https://zenodo.org/records/7614764/files/distemist_zenodo.zip?download=1"

from requests import get
from zipfile import ZipFile
from io import BytesIO
DATASET_PATH = "/gpfs/projects/bsc14/.nlp4bio"
# CACHE_DIR = os.path.join(DATASET_PATH, "cache")

os.makedirs(DATASET_PATH, exist_ok=True)
response = get(url)
zip_file = ZipFile(BytesIO(response.content))
zip_file.extractall(DATASET_PATH)


In [3]:
from nlp4bia.datasets.Dataset import Dataset

class Distemist(Dataset):
    
    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        
        train_path = os.path.join(self.path, "training/subtrack2_linking")
        texts_train_path = os.path.join(self.path, "training/text_files")
        test_path = os.path.join(self.path, "test_annotated/subtrack2_linking")
        texts_test_path = os.path.join(self.path, "test_annotated/text_files")
        
        df_train = pd.DataFrame()
        for path in os.listdir(train_path):
            df_i = pd.read_csv(os.path.join(train_path, path), sep="\t")
            df_train = pd.concat([df_train, df_i])
        
        df_test = pd.DataFrame()
        for path in os.listdir(test_path):
            df_i = pd.read_csv(os.path.join(test_path, path), sep="\t")
            df_test = pd.concat([df_test, df_i])
        
        df_train["split"] = "train"
        df_test["split"] = "test"
        
        df = pd.concat([df_train, df_test], ignore_index=True)
        
        df_texts = self.get_texts(texts_train_path, texts_test_path)
        df = df.merge(df_texts, on="filename", how="left")
        
        assert df.duplicated(subset=["filename", "mark"]).sum() == 0, "There are duplicated filename+marks"
        
        return df
    
    @staticmethod
    def get_texts(*paths, extension=".txt"):
        '''Get texts from text_files
        Input: paths: sequence of paths to text_files
        Output: DataFrame with columns: filename, text
        '''
        ls_texts_path = []
        for path in paths:
            ls_texts_path_i = []
            # For each main path, extract the filenames
            for filename in os.listdir(path):
                ls_texts_path_i.append((path, filename))
            
            # Append the list of filenames to the main list
            ls_texts_path.extend(ls_texts_path_i)
        
        # Retrieve the text from each file and create tuples with the filename and the content
        ls_texts = [(filename, open(os.path.join(path, filename)).read()) for (path, filename) in ls_texts_path]
        
        df_texts = pd.DataFrame(ls_texts, columns=["filename", "text"])
        df_texts["filename"] = df_texts["filename"].str.replace(extension, "") # remove the extension
                
        return df_texts
    
    def preprocess_data(self):
        print("preprocessing data...")

In [4]:
os.listdir(DATASET_PATH)

['cache', 'distemist_zenodo']

In [5]:
dist_path = os.path.join(DATASET_PATH, "distemist_zenodo")
dm = Distemist(dist_path, lang="es")

In [6]:
df_dm = dm.load_data()
df_dm.head()

,filename,mark,label,off0,off1,span,code,semantic_rel,split,text
0,es-S0210-56912007000900007-3,T1,ENFERMEDAD,164,166,DM,73211009,EXACT,train,Mujer de 74 años que ingresó en el hospital po...
1,es-S0210-56912007000900007-3,T2,ENFERMEDAD,362,376,deshidratación,34095006,EXACT,train,Mujer de 74 años que ingresó en el hospital po...
2,es-S0210-56912007000900007-3,T3,ENFERMEDAD,575,590,hiperamilasemia,275739007,EXACT,train,Mujer de 74 años que ingresó en el hospital po...
3,es-S0210-56912007000900007-3,T4,ENFERMEDAD,715,733,pancreatitis aguda,197456007,EXACT,train,Mujer de 74 años que ingresó en el hospital po...
4,es-S0210-56912007000900007-3,T5,ENFERMEDAD,1402,1459,formación polipoidea sésil situada junto al es...,88580009,EXACT,train,Mujer de 74 años que ingresó en el hospital po...


In [5]:
print(dm)


Distemist(path=/gpfs/projects/bsc14/storage/corpus/distemist_sp_ca_en_fr_it_ro_po, version=raw, lang=es)


In [38]:
df_dm["text"].isnull().mean()

0.6640806826997673

In [ ]:
train_path = os.path.join(dist_path, "raw", "training/subtrack2_linking")

df_train = pd.DataFrame()
for path in os.listdir(train_path):
    df_i = pd.read_csv(os.path.join(train_path, path), sep="\t")
    df_train = pd.concat([df_train, df_i])

df_train["split"] = "train"


,filename,mark,label,off0,off1,span,code,semantic_rel
0,es-S0210-56912007000900007-3,T1,ENFERMEDAD,164,166,DM,73211009,EXACT
1,es-S0210-56912007000900007-3,T2,ENFERMEDAD,362,376,deshidratación,34095006,EXACT
2,es-S0210-56912007000900007-3,T3,ENFERMEDAD,575,590,hiperamilasemia,275739007,EXACT
3,es-S0210-56912007000900007-3,T4,ENFERMEDAD,715,733,pancreatitis aguda,197456007,EXACT
4,es-S0210-56912007000900007-3,T5,ENFERMEDAD,1402,1459,formación polipoidea sésil situada junto al es...,88580009,EXACT
...,...,...,...,...,...,...,...,...
1507,es-S0365-66912008000100007-1,T5,ENFERMEDAD,914,957,malformación colobomatosa del nervio óptico,77157004,NARROW
1508,es-S0365-66912008000100007-1,T6,ENFERMEDAD,960,984,lagunas coriorretinianas,302893000,NARROW
1509,es-S0365-66912008000100007-1,T7,ENFERMEDAD,462,496,anomalías de la migración neuronal,253146009,EXACT
1510,es-S0365-66912008000100007-1,T8,ENFERMEDAD,837,855,afectación macular,312999006,EXACT


In [4]:
dm = Distemist(dist_path, lang="es")

TypeError: Can't instantiate abstract class Distemist with abstract methods load_data, preprocess_data

In [8]:
df_distemist1 = pd.read_csv(os.path.join(ds.full_path, 'training/subtrack2_linking/', 'distemist_subtrack2_training1_linking.tsv'), sep='\t', dtype={'code': str})
df_distemist2 = pd.read_csv(os.path.join(ds.full_path, 'training/subtrack2_linking/', 'distemist_subtrack2_training2_linking.tsv'), sep='\t', dtype={'code': str})
df_distemist_test = pd.read_csv(os.path.join(ds.full_path, 'test_annotated/subtrack2_linking/', 'distemist_subtrack2_test_linking.tsv'), sep='\t', dtype={'code': str})
df_distemist = pd.concat([df_distemist1, df_distemist2, df_distemist_test], ignore_index=True)

print("Original shape:", df_distemist.shape, "\n")
print("Duplicates by [filename, off0, off1]:", df_distemist[df_distemist.duplicated(subset=["filename", "off0", "off1"], keep=False)].shape[0], "\n")

# print(df_distemist[~df_distemist['code'].str.match(r'^\d+$')].to_markdown(), "\n")
print(df_distemist[~df_distemist.code.str.match(r'(\d+\+?)+')].to_markdown(), "\n")

df_distemist = df_distemist[df_distemist.code.str.match(r'(\d+\+?)+')]
print("Shape after removing non-numeric codes:", df_distemist.shape)

print("Null content:", df_distemist.isnull().sum().sum(), "\n")

df_distemist.drop(columns=["mark"], inplace=True)
df_distemist = df_distemist[["filename", "label", "off0", "off1", "span", "code", "semantic_rel"]]

Original shape: (7734, 8) 

Duplicates by [filename, off0, off1]: 0 

|      | filename                     | mark   | label      |   off0 |   off1 | span   | code   | semantic_rel   |
|-----:|:-----------------------------|:-------|:-----------|-------:|-------:|:-------|:-------|:---------------|
| 4511 | es-S1130-05582017000100044-2 | T9     | ENFERMEDAD |    954 |    957 | SPM    | NOMAP  | NOMAP          | 

Shape after removing non-numeric codes: (7733, 8)
Null content: 0 

